In [2]:
# ssl/certifi: fix HTTPS cert verification on macOS python.org installs
# requests: makes the HTTP call so we control the headers
# pandas: parses HTML tables into DataFrames
import ssl, certifi, requests
import pandas as pd
from io import StringIO

# Point Python's default HTTPS context at certifi's CA bundle.
# The python.org installer doesn't hook into the macOS system keychain,
# so without this, HTTPS requests can fail with SSL: CERTIFICATE_VERIFY_FAILED.
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

# The page holding both tables we want: current constituents + historical changes log.
# %26 is the URL-encoded "&" in "S&P".
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Wikipedia returns 403 Forbidden to clients with a default/absent User-Agent.
# Presenting a real browser UA string gets us served normally.
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"}

# Fetch the page ourselves (rather than letting pandas fetch it) so our headers apply.
resp = requests.get(WIKI_URL, headers=HEADERS)

# requests does NOT raise on 4xx/5xx — it just returns a response with a bad status.
# This converts a bad status into an exception, so we fail loudly here at the fetch
# instead of confusingly later at the parse with an unhelpful "no tables found".
resp.raise_for_status()

# pandas 3.0 removed support for passing literal HTML strings — a bare string is
# interpreted as a file path. StringIO wraps the string in a file-like object,
# making it unambiguous that this is content, not a location.
tables = pd.read_html(StringIO(resp.text))


# Inspect what we actually got, rather than assuming the tables we want sit at
# fixed indices 0 and 1 — the page also contains sector navboxes and other tables,
# and their order can shift with any Wikipedia edit.
print(f"{len(tables)} tables found\n")
for i, t in enumerate(tables):
    # enumerate() yields (index, item) pairs, so we can report each table's position
    print(f"--- table {i}: shape {t.shape} ---")  # .shape is (rows, columns)
    print(list(t.columns))                        # the real column names, to match on later
    print()


3 tables found

--- table 0: shape (503, 8) ---
['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']

--- table 1: shape (406, 6) ---
[('Effective Date', 'Effective Date'), ('Added', 'Ticker'), ('Added', 'Security'), ('Removed', 'Ticker'), ('Removed', 'Security'), ('Reason', 'Reason')]

--- table 2: shape (11, 2) ---
['vteS&P 500 companies', 'vteS&P 500 companies.1']



In [3]:
# Grab the two tables we care about. (Indices 0 and 1 happen to be right here, but
# we confirmed that by inspection rather than assuming it — and we'll replace this
# with signature-based matching when this moves into src/.)
current = tables[0]
changes = tables[1]

# Flatten the two-level MultiIndex into single-level names.
# ('Added', 'Ticker') -> 'added_ticker', but ('Effective Date', 'Effective Date')
# -> 'effective_date' rather than a stuttering 'effective_date_effective_date'.
def flatten(col):
    top, bottom = col
    name = top if top == bottom else f"{top} {bottom}"
    return name.strip().lower().replace(" ", "_")

changes.columns = [flatten(c) for c in changes.columns]
print(list(changes.columns))

# Look at both ends: the newest changes and the oldest ones the table contains.
display(changes.head(10))
display(changes.tail(10))


['effective_date', 'added_ticker', 'added_security', 'removed_ticker', 'removed_security', 'reason']


,effective_date,added_ticker,added_security,removed_ticker,removed_security,reason
0,"June 30, 2026",NaN,NaN,CAG,Conagra Brands,Market capitalization change.[5]
1,"June 29, 2026",HONA,Honeywell Aerospace,NaN,NaN,S&P 500 constituent Honeywell spun off Honeywe...
2,"June 22, 2026",MRVL,Marvell Technology,POOL,Pool Corporation,Market capitalization change.[6]
3,"June 22, 2026",FLEX,Flex Ltd.,CPB,Campbell's,Market capitalization change.[6]
4,"June 2, 2026",NaN,NaN,EPAM,EPAM Systems,Market capitalization change.[7]
5,"June 1, 2026",FDXF,FedEx Freight,NaN,NaN,S&P 500 & 100 constituent FedEx Corp. spun off...
6,"May 7, 2026",VEEV,Veeva Systems,CTRA,Coterra Energy,S&P 500 constituent Devon Energy Corp. acquire...
7,"April 9, 2026",CASY,Casey's,HOLX,Hologic,Blackstone Inc. and TPG Inc. acquired Hologic....
8,"March 23, 2026",VRT,Vertiv,MTCH,Match Group,Market capitalization change.[11]
9,"March 23, 2026",LITE,Lumentum,MOH,Molina Healthcare,Market capitalization change.[11]


,effective_date,added_ticker,added_security,removed_ticker,removed_security,reason
396,"December 8, 1999",YHOO,Yahoo!,LDW,Laidlaw,Market capitalization change.[278]
397,"June 9, 1999",WLP,Wellpoint,HPH,Harnischfeger Industries,Harnischfeger filed for bankruptcy.[279]
398,"April 12, 1999",ACT,Actavis,NaN,NaN,Actavis plc (NYSE:ACT) added to S&P 500
399,"December 11, 1998",FSR,Firstar,AN,Amoco,BP purchased Amoco.[280]
400,"December 11, 1998",CCL,Carnival Corporation,GRN,General Re,Berkshire Hathaway purchased General Re.[280]
401,"December 11, 1998",CPWR,Compuware,SUN,SunAmerica,AIG purchased SunAmerica.[280]
402,"June 17, 1997",CCR,Countrywide Credit Industries,USL,USLife,AIG acquired USLife.[281]
403,"September 30, 1994",NCC,National City,MCK,McKesson,McKesson sold PCS Health Services to Eli Lilly...
404,"July 1, 1976",BUD,Anheuser Busch,HNG,Houston Natural Gas,Major restructuring of S&P 500 to have fewer i...
405,"July 1, 1976",DIS,The Walt Disney Company,AYE,Allegheny Energy,Major restructuring of S&P 500 to have fewer i...


In [4]:
# Parse the human-readable dates ("June 30, 2026") into real timestamps.
# errors="coerce" turns anything unparseable into NaT rather than raising, so a few
# malformed rows don't kill the whole call — but we then COUNT them, because silently
# dropping bad dates is how you end up with a quietly incomplete universe.
changes["effective_date"] = pd.to_datetime(
    changes["effective_date"], format="mixed", errors="coerce"
)
print("unparsed dates:", changes["effective_date"].isna().sum())

# How many recorded changes per year? This is the coverage audit.
per_year = changes["effective_date"].dt.year.value_counts().sort_index()
print(per_year.to_string())


unparsed dates: 0
effective_date
1976     2
1994     1
1997     1
1998     3
1999     3
2000     7
2003     1
2005     2
2006     1
2007    11
2008     8
2009    13
2010    11
2011    19
2012    18
2013    19
2014    16
2015    29
2016    30
2017    29
2018    23
2019    24
2020    20
2021    20
2022    22
2023    18
2024    21
2025    21
2026    13


In [5]:
# count() ignores NaN, so this tallies only rows that actually recorded an add / a remove.
flow = changes.groupby(changes["effective_date"].dt.year).agg(
    adds=("added_ticker", "count"),
    removes=("removed_ticker", "count"),
)
flow["net"] = flow["adds"] - flow["removes"]

# Walk backward from today's 503. Undoing a year's changes means subtracting its net
# effect, so the implied size at the START of year Y is 503 minus the net of every
# change from Y onward — a reverse cumulative sum.
net_desc = flow["net"].sort_index(ascending=False)   # 2026 -> 1976
flow["implied_size_start_of_year"] = (503 - net_desc.cumsum()).sort_index()

print(flow.to_string())


                adds  removes  net  implied_size_start_of_year
effective_date                                                
1976               2        2    0                         499
1994               1        1    0                         499
1997               1        1    0                         499
1998               3        3    0                         499
1999               3        2    1                         499
2000               7        7    0                         500
2003               1        1    0                         500
2005               2        2    0                         500
2006               1        1    0                         500
2007              11       11    0                         500
2008               8        8    0                         500
2009              12       12    0                         500
2010              11       11    0                         500
2011              19       19    0                     

In [6]:
API_URL = "https://en.wikipedia.org/w/api.php"
PAGE_TITLE = "List of S&P 500 companies"

# Wikimedia asks API clients to identify themselves with a descriptive User-Agent
# including contact info — a browser UA string is fine for page scraping but
# considered impolite for API use, and can get you rate-limited.
API_HEADERS = {"User-Agent": "capm-portfolio/0.1 (research project; https://github.com/kc2998/capm-portfolio)"}

def revision_at(date_iso):
    """Return (revid, timestamp) of the last revision at or before date_iso."""
    params = {
        "action": "query", "prop": "revisions", "titles": PAGE_TITLE,
        "rvlimit": 1,
        "rvdir": "older",        # walk backward in time from rvstart
        "rvstart": date_iso,     # e.g. "2015-06-30T00:00:00Z"
        "rvprop": "ids|timestamp",
        "format": "json", "formatversion": 2,
    }
    r = requests.get(API_URL, params=params, headers=API_HEADERS)
    r.raise_for_status()
    revs = r.json()["query"]["pages"][0].get("revisions", [])
    return (revs[0]["revid"], revs[0]["timestamp"]) if revs else (None, None)

def revision_html(revid):
    """Fetch the rendered HTML of a specific revision."""
    params = {"action": "parse", "oldid": revid, "prop": "text",
              "format": "json", "formatversion": 2}
    r = requests.get(API_URL, params=params, headers=API_HEADERS)
    r.raise_for_status()
    return r.json()["parse"]["text"]

revid, ts = revision_at("2015-06-30T00:00:00Z")
print("revision:", revid, "timestamp:", ts)

snapshot_tables = pd.read_html(StringIO(revision_html(revid)))
print(f"\n{len(snapshot_tables)} tables in the 2015 revision\n")
for i, t in enumerate(snapshot_tables):
    print(f"--- table {i}: shape {t.shape} ---")
    print(list(t.columns))
    print()


revision: 669156533 timestamp: 2015-06-29T08:14:14Z

2 tables in the 2015 revision

--- table 0: shape (502, 8) ---
['Ticker symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub Industry', 'Address of Headquarters', 'Date first added', 'CIK']

--- table 1: shape (104, 6) ---
[0, 1, 2, 3, 4, 5]



In [7]:
import time

def find_constituents(tabs):
    """Pick the table that looks like a constituents list: has a ticker-ish column.
    Matching on a column signature rather than a fixed index is what lets one parser
    work across a decade of Wikipedia reformatting."""
    for i, t in enumerate(tabs):
        cols = [str(c).lower() for c in t.columns]
        if any("symbol" in c or "ticker" in c for c in cols):
            return i, t
    return None, None

for d in ["2006-06-30", "2008-06-30", "2010-06-30", "2012-06-30",
          "2014-06-30", "2018-06-30", "2022-06-30"]:
    try:
        revid, ts = revision_at(f"{d}T00:00:00Z")
        if revid is None:
            print(f"{d}: no revision found")
            continue
        tabs = pd.read_html(StringIO(revision_html(revid)))
    except Exception as e:
        # Broad except is right *here*: the whole point is to survey which dates work
        # and which don't. We print the error type so a failure is still a data point.
        print(f"{d}: FAILED — {type(e).__name__}: {e}")
        continue

    i, t = find_constituents(tabs)
    if t is None:
        print(f"{d}: rev {revid} ({ts[:10]}) — {len(tabs)} tables, none ticker-like")
    else:
        print(f"{d}: rev {revid} ({ts[:10]}) — table {i}, {t.shape[0]} rows | {list(t.columns)}")
    time.sleep(0.5)


2006-06-30: FAILED — ValueError: No tables found
2008-06-30: rev 220621121 (2008-06-20) — table 0, 500 rows | ['Ticker symbol', 'Company', 'SEC filings', 'GICS Sector']
2010-06-30: rev 360899452 (2010-05-08) — table 0, 500 rows | ['Ticker symbol', 'Company', 'SEC filings', 'GICS Sector']
2012-06-30: rev 498154602 (2012-06-18) — table 0, 500 rows | ['Ticker symbol', 'Company', 'SEC filings', 'GICS Sector', 'Address of Headquarters']
2014-06-30: rev 614647412 (2014-06-27) — table 0, 501 rows | ['Ticker symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub Industry', 'Address of Headquarters', 'Date first added', 'CIK']
2018-06-30: rev 848091676 (2018-06-29) — table 0, 505 rows | ['Ticker symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub Industry', 'Location', 'Date first added[3][4]', 'CIK', 'Founded']
2022-06-30: rev 1095558369 (2022-06-29) — table 0, 503 rows | ['Symbol', 'Security', 'SEC filings', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date fir